# SentimentFlow — Notebook 01: Análisis de Sentimiento

Versión interactiva del job de PySpark que procesa un lote de reseñas.
En el pipeline automático, este mismo código se ejecuta vía Airflow (`spark/sentiment_analysis.py`).

**Flujo:**
1. Descarga el fichero `.jsonl` desde MinIO (bucket `raw-reviews`).
2. Lee y limpia los datos con PySpark.
3. Aplica análisis de sentimiento con **VADER**.
4. Escribe los resultados en PostgreSQL (`reviews_sentiment`).
5. Sube el Parquet procesado a MinIO (`processed-reviews`).

In [ ]:
import os

# ─── Configuración — ajustar si es necesario ───────────────────────────────
MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',   'http://minio:9000')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
MINIO_BUCKET_RAW = os.getenv('MINIO_BUCKET_RAW', 'raw-reviews')
MINIO_BUCKET_PRO = os.getenv('MINIO_BUCKET_PROCESSED', 'processed-reviews')

PG_HOST = os.getenv('POSTGRES_HOST', 'postgres')
PG_PORT = int(os.getenv('POSTGRES_PORT', '5432'))
PG_DB   = os.getenv('POSTGRES_DB',   'reviewsdb')
PG_USER = os.getenv('POSTGRES_USER', 'postgres')
PG_PASS = os.getenv('POSTGRES_PASSWORD', 'postgres')

# Cambiar por el nombre del fichero que se quiere procesar
INPUT_FILE = 'batch_20260501T120000_0100.jsonl'

print(f'MinIO endpoint : {MINIO_ENDPOINT}')
print(f'PostgreSQL     : {PG_HOST}:{PG_PORT}/{PG_DB}')
print(f'Fichero de entrada: {INPUT_FILE}')

In [ ]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

# Listar ficheros disponibles en raw-reviews
resp = s3.list_objects_v2(Bucket=MINIO_BUCKET_RAW)
ficheros = [o['Key'] for o in resp.get('Contents', [])]
print(f'Ficheros en raw-reviews ({len(ficheros)}):')
for f in ficheros:
    print(' -', f)

In [ ]:
import tempfile, os

tmp_dir = tempfile.mkdtemp(prefix='sentimentflow_nb_')
local_input = os.path.join(tmp_dir, INPUT_FILE)

s3.download_file(MINIO_BUCKET_RAW, INPUT_FILE, local_input)
print(f'Descargado: {local_input} ({os.path.getsize(local_input)/1024:.1f} KB)')

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName('SentimentFlow-Notebook01')
    .master('local[*]')
    .config('spark.driver.memory', '1g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

schema = StructType([
    StructField('review_id',    StringType(),  False),
    StructField('product_id',   StringType(),  False),
    StructField('product_name', StringType(),  True),
    StructField('category',     StringType(),  True),
    StructField('user_id',      StringType(),  True),
    StructField('rating',       IntegerType(), True),
    StructField('review_text',  StringType(),  True),
    StructField('country',      StringType(),  True),
    StructField('timestamp',    StringType(),  True),
])

df_raw = spark.read.schema(schema).json(local_input)
print(f'Registros leídos: {df_raw.count()}')
df_raw.show(5, truncate=60)

In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def clean_text(text):
    if not text:
        return ''
    text = text.lower().strip()
    text = re.sub(r'[^\w\s.,!?áéíóúüñ]', ' ', text)
    return re.sub(r'\s+', ' ', text)

clean_udf = F.udf(clean_text, StringType())

df_clean = (
    df_raw
    .withColumn('review_text_clean', clean_udf(F.col('review_text')))
    .withColumn('event_ts', F.to_timestamp(F.col('timestamp')))
    .filter(F.col('review_id').isNotNull())
    .filter(F.col('rating').between(1, 5))
    .dropDuplicates(['review_id'])
)
print(f'Registros tras limpieza: {df_clean.count()}')

In [ ]:
from pyspark.sql.types import DoubleType
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# NOTA: VADER está entrenado en inglés. Los textos en español tenderán
# a puntuaciones neutras. Para producción real usar pysentimiento.
analyzer = SentimentIntensityAnalyzer()

def get_compound(text):
    return float(analyzer.polarity_scores(text or '')['compound'])

def get_label(score):
    if score is None: return 'Neutro'
    if score >= 0.05:  return 'Positivo'
    if score <= -0.05: return 'Negativo'
    return 'Neutro'

compound_udf = F.udf(get_compound, DoubleType())
label_udf    = F.udf(get_label,    StringType())

from datetime import datetime, timezone
df_sentiment = (
    df_clean
    .withColumn('sentiment_score', compound_udf(F.col('review_text_clean')))
    .withColumn('sentiment_label', label_udf(F.col('sentiment_score')))
    .withColumn('processed_at', F.lit(datetime.now(timezone.utc).isoformat()).cast('timestamp'))
)

print('Distribución de sentimiento:')
df_sentiment.groupBy('sentiment_label').count().orderBy('count', ascending=False).show()

print('Sentimiento medio por categoría:')
df_sentiment.groupBy('category').agg(
    F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
    F.round(F.avg('rating'), 2).alias('avg_rating'),
    F.count('*').alias('total'),
).show()

In [ ]:
import psycopg2, psycopg2.extras

OUTPUT_COLS = [
    'review_id', 'product_id', 'product_name', 'category',
    'user_id', 'rating', 'review_text_clean', 'country',
    'event_ts', 'sentiment_score', 'sentiment_label', 'processed_at',
]
rows = df_sentiment.select(*OUTPUT_COLS).collect()

conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS)
with conn.cursor() as cur:
    psycopg2.extras.execute_values(
        cur,
        """
        INSERT INTO reviews_sentiment
          (review_id, product_id, product_name, category, user_id,
           rating, review_text, country, event_ts,
           sentiment_score, sentiment_label, processed_at)
        VALUES %s
        ON CONFLICT (review_id) DO NOTHING
        """,
        [(r.review_id, r.product_id, r.product_name, r.category,
          r.user_id, r.rating, r.review_text_clean, r.country,
          r.event_ts, r.sentiment_score, r.sentiment_label, r.processed_at)
         for r in rows],
    )
conn.commit()
conn.close()
print(f'✓ {len(rows)} filas insertadas en PostgreSQL.')

In [ ]:
import glob, shutil
from datetime import datetime, timezone

parquet_dir = os.path.join(tmp_dir, 'output_parquet')
df_sentiment.select(*OUTPUT_COLS).coalesce(1).write.mode('overwrite').parquet(parquet_dir)

run_date = datetime.now(timezone.utc).strftime('%Y/%m/%d')
base_name = INPUT_FILE.replace('.jsonl', '')

for pf in glob.glob(os.path.join(parquet_dir, '*.parquet')):
    key = f'{run_date}/{base_name}.parquet'
    s3.upload_file(pf, MINIO_BUCKET_PRO, key)
    print(f'✓ Parquet subido: processed-reviews/{key}')

shutil.rmtree(tmp_dir, ignore_errors=True)
spark.stop()
print('Notebook completado.')